# Notebook: LRD4 Colab helper

This notebook prepares the environment and runs quick tests for the D1–D2–D3 model: H(z) comparison and rotation-curve demo. Follow cells in order.

## 1. Configurer l'environnement Colab

Ce bloc vérifie la version de Python/pip et donne des instructions pour choisir runtime (GPU/TPU) dans Colab (Menu Runtime > Change runtime type).

In [ ]:
# Check Python and pip versions
import sys, subprocess
print('Python', sys.version)
print('pip version:')
subprocess.run([sys.executable, '-m', 'pip', '--version'])

## 2. Monter Google Drive

Monter Google Drive pour sauvegarder résultats et lire fichiers depuis Drive.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

base_dir = '/content/drive/MyDrive/ColabResults/LRD4'
os.makedirs(base_dir, exist_ok=True)
print('Base dir:', base_dir)

## 3. Installer dépendances et versions

Installer paquets utiles et afficher leurs versions. Modifiez la liste selon vos besoins.

In [ ]:
# Installer packages requis (exécuter dans Colab)
!pip install numpy matplotlib pandas openpyxl papermill scikit-learn --quiet

import pkg_resources
for pkg in ['numpy','matplotlib','pandas','openpyxl','papermill','sklearn']:
    try:
        print(pkg, pkg_resources.get_distribution(pkg).version)
    except Exception as e:
        print(pkg, 'not found')

## 4. Importer et organiser les fichiers du projet

Vous pouvez cloner un dépôt Git, ou copier les fichiers depuis Drive. Ici on montre comment copier `scripts/quick_test_model.py` depuis Drive si vous l'avez uploadé.

In [ ]:
# Exemple: copier un script depuis Drive (si vous l'avez uploadé)
import shutil
src = '/content/drive/MyDrive/ColabResults/LRD4/quick_test_model.py'
dst = '/content/quick_test_model.py'
if os.path.exists(src):
    shutil.copy(src, dst)
    print('Copied', src)
else:
    print('No script found in Drive; you can upload via the left pane or use files.upload()')

# Ajouter le répertoire courant au path
import sys
sys.path.append('/content')


## 5. Charger et prétraiter les données

Charger le fichier Excel `ListeDesObservationInexpliquer.xlsx` s'il est uploadé dans Drive ou dans l'environnement Colab.

In [ ]:
import pandas as pd
from google.colab import files

# Try loading from Drive path used earlier
drive_path = '/content/drive/MyDrive/ColabResults/LRD4/ListeDesObservationInexpliquer.xlsx'
if os.path.exists(drive_path):
    df = pd.read_excel(drive_path)
    print('Loaded from Drive:', drive_path)
else:
    print('Not found in Drive — use upload dialog')
    uploaded = files.upload()
    for fn in uploaded:
        if fn.lower().endswith('.xlsx'):
            df = pd.read_excel(fn)
            print('Loaded', fn)
            break

print('Dataframe shape:', df.shape)
df.head()

## 6. Exploration et visualisation des données

Affichage des statistiques basiques et histogrammes pour les colonnes numériques.

In [ ]:
import matplotlib.pyplot as plt

num_cols = df.select_dtypes(include=['number']).columns.tolist()
print('Numeric columns:', num_cols)
if len(num_cols)>0:
    df[num_cols].hist(figsize=(10,6))
    plt.tight_layout()
    plt.show()


## 7. Définir, entraîner et sauvegarder un modèle (demo numerics)

Ici nous ré-utilisons la logique du script rapide pour H(z) et courbe de rotation et sauvegardons les figures.

In [ ]:
# Parameters (modifiable)
Omega_emergent = 0.25
Omega_act0 = 0.7
z_star = 0.46
dz = 0.1
rho0 = 0.01

# Cosmology functions
import numpy as np
H0 = 70.0
Omega_b = 0.05
Omega_m_LCDM = 0.3
Omega_L_LCDM = 0.7

def H_LCDM(z):
    return H0 * np.sqrt(Omega_m_LCDM*(1+z)**3 + Omega_L_LCDM)

def H_LRD(z):
    rho_emergent = Omega_emergent * (1+z)**3
    rho_act = Omega_act0 * np.tanh((z - z_star)/dz)
    return H0 * np.sqrt(Omega_b*(1+z)**3 + rho_emergent + np.maximum(rho_act, 0.0))

z = np.linspace(0,2,201)

# Plot and save
import matplotlib.pyplot as plt
plt.figure()
plt.plot(z, H_LCDM(z)/H0, label='LCDM')
plt.plot(z, H_LRD(z)/H0, label='LRD-like')
plt.xlabel('z')
plt.ylabel('H(z)/H0')
plt.legend()
plt.grid(True)
fn1 = os.path.join(base_dir, 'H_vs_z_colab.png')
plt.savefig(fn1, dpi=150)
plt.show()
print('Saved', fn1)

# Rotation curve demo
G = 4.30091727003628e-6
M_disk = 5e10
r_s = 3.0
r = np.linspace(0.1,30,300)
M_enclosed_disk = M_disk * (1 - np.exp(-r/r_s)*(1 + r/r_s))
V_disk = np.sqrt(G * M_enclosed_disk / r)
M_enclosed_em = 4*np.pi * rho0 * 1.0**2 * r
V_em = np.sqrt(G * M_enclosed_em / r)
V_tot = np.sqrt(V_disk**2 + V_em**2)

plt.figure()
plt.plot(r, V_disk, label='Baryons')
plt.plot(r, V_em, label='Emergent halo')
plt.plot(r, V_tot, label='Total')
plt.xlabel('r [kpc]')
plt.ylabel('v_c [km/s]')
plt.legend()
plt.grid(True)
fn2 = os.path.join(base_dir, 'rotation_curve_colab.png')
plt.savefig(fn2, dpi=150)
plt.show()
print('Saved', fn2)

# Save a small CSV summary
import pandas as pd
summary = pd.DataFrame({'z': z, 'H_LCDM': H_LCDM(z), 'H_LRD': H_LRD(z)})
summary_fn = os.path.join(base_dir, 'H_summary.csv')
summary.to_csv(summary_fn, index=False)
print('Saved', summary_fn)


## 8. Évaluer le modèle et générer métriques

Ce bloc est un point d'entrée pour ajouter évaluations plus avancées (SN Ia fit, BAO, etc.). Pour l'instant on sauvegarde les résultats générés ci-dessus.

## 9. Optimiser l'entraînement (GPU, mixed precision, checkpoints)

Instructions et utilitaires pour activer mixed precision et gérer checkpoints (si vous utilisez TF/PyTorch).

## 10. Tests unitaires et exécution reproductible

Exemple: fixation du seed pour numpy et pandas, et suggestion de tests pytest pour fonctions critiques.

In [ ]:
import numpy as np
np.random.seed(42)

# Example pytest suggestion (not executed here):
# def test_H_LRD():
#     assert H_LRD(0) > 0

print('Seed fixé pour numpy')

## 11. Exporter notebook et artefacts

Exemples pour exporter le notebook en HTML et télécharger les artefacts sauvegardés.

In [ ]:
# Exporter en HTML
import nbformat
from nbconvert import HTMLExporter

nb = nbformat.read('colab_lrd4.ipynb', as_version=4)
html_exporter = HTMLExporter()
(body, resources) = html_exporter.from_notebook_node(nb)
open('/content/drive/MyDrive/ColabResults/LRD4/colab_lrd4.html','w', encoding='utf-8').write(body)
print('Exporté vers Drive')